# Colab Training Notebook

This notebook is the Colab entrypoint for training the custom anchor-based detector from this repository.

Runtime requirement: use `GPU` in Colab. A T4 GPU is appropriate. Do not select TPU for this PyTorch codebase.


## Usage

1. In Colab, set `Runtime -> Change runtime type -> Hardware accelerator -> GPU`.
2. Fill in `REPO_URL` and optionally `BRANCH_NAME` below.
3. Run the notebook top to bottom.
4. Outputs, plots, metrics, and checkpoints can be written to Google Drive.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
BRANCH_NAME = "main"
WORKDIR = Path("/content/Small_Obj_detection")
USE_DRIVE_OUTPUTS = True
DRIVE_OUTPUTS_DIR = Path("/content/drive/MyDrive/small_obj_detection_runs")

FOLD_INDEX = 0
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
FREEZE_BACKBONE = False
RUN_NAME = f"colab_fold{FOLD_INDEX}_baseline"


In [ ]:
import os
import shutil
import subprocess
import sys

if "google.colab" not in sys.modules:
    raise RuntimeError("This notebook is intended to run inside Google Colab.")
if "<your-user>" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository before running this notebook.")

if USE_DRIVE_OUTPUTS:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.check_call(["git", "clone", "--branch", BRANCH_NAME, REPO_URL, str(WORKDIR)])
os.chdir(WORKDIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print(f"Repo ready at: {WORKDIR}")


In [ ]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download("daenys2000/small-object-dataset"))
print(f"Dataset path: {dataset_path}")


In [ ]:
import json
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

from src.data.dataset import build_fold_tile_datasets, detection_collate_fn
from src.data.eda import build_box_level_table, save_box_level_table, save_eda_summary, summarize_eda
from src.data.folds import make_image_level_folds, plot_fold_balance, save_fold_assignments, save_fold_summary, summarize_folds, validate_fold_assignments
from src.data.indexing import build_labeled_records, save_records_table
from src.data.tiling import build_tile_tables, save_tile_box_table, save_tile_summary, save_tile_table, summarize_tiles, validate_tile_tables
from src.data.verification import audit_annotations, filter_trusted_records, save_annotation_audit, save_annotation_audit_summary, summarize_annotation_audit
from src.models.anchors import AnchorGenerator
from src.models.detector import SmallObjectDetector
from src.train.engine import fit_detector, save_history
from src.utils import ExperimentConfig, set_seed

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("GPU runtime not detected. In Colab, switch to a GPU runtime before training.")

config = ExperimentConfig()
config.paths.root = WORKDIR
config.paths.dataset_root = dataset_path
if USE_DRIVE_OUTPUTS:
    config.paths.outputs_dir = DRIVE_OUTPUTS_DIR / RUN_NAME
    config.paths.reports_dir = config.paths.outputs_dir / "reports"
    config.paths.metrics_dir = config.paths.outputs_dir / "metrics"
    config.paths.plots_dir = config.paths.outputs_dir / "plots"
    config.paths.checkpoints_dir = config.paths.outputs_dir / "checkpoints"
config.paths.ensure()

config.train.batch_size = BATCH_SIZE
config.train.epochs = EPOCHS
config.train.learning_rate = LEARNING_RATE
config.train.weight_decay = WEIGHT_DECAY
config.train.num_workers = NUM_WORKERS

print(f"Device: {device}")
print(f"Outputs dir: {config.paths.outputs_dir}")
print(f"Dataset root: {config.paths.validate_dataset_root()}")


In [ ]:
records = build_labeled_records(config.paths.dataset_root)
save_records_table(records, config.paths.metrics_dir / "colab_records.csv")

audit_df = audit_annotations(records)
save_annotation_audit(audit_df, config.paths.metrics_dir / "colab_annotation_audit.csv")
audit_summary = summarize_annotation_audit(audit_df)
save_annotation_audit_summary(audit_summary, config.paths.reports_dir / "colab_annotation_audit_summary.json")

trusted_records = filter_trusted_records(records, audit_df)
save_records_table(trusted_records, config.paths.metrics_dir / "colab_trusted_records.csv")

box_level = build_box_level_table(trusted_records)
save_box_level_table(box_level, config.paths.metrics_dir / "colab_box_level_eda.csv")
eda_summary = summarize_eda(trusted_records, box_level)
save_eda_summary(eda_summary, config.paths.reports_dir / "colab_eda_summary.json")

assignments = make_image_level_folds(trusted_records, num_folds=config.data.num_folds, random_state=config.data.fold_seed)
validate_fold_assignments(trusted_records, assignments)
save_fold_assignments(assignments, config.paths.metrics_dir / "colab_fold_assignments.csv")
fold_summary = summarize_folds(trusted_records, assignments)
save_fold_summary(fold_summary, config.paths.reports_dir / "colab_fold_summary.json")
plot_fold_balance(assignments, config.paths.plots_dir / "colab_fold_balance.png")

tile_table, tile_box_table = build_tile_tables(
    trusted_records,
    assignments,
    tile_size=config.tiles.tile_size,
    overlap=config.tiles.overlap,
    coordinate_mode=config.data.bbox_coordinate_mode,
    min_box_area=config.tiles.min_box_area,
    keep_empty_tiles=config.tiles.keep_empty_tiles,
)
validate_tile_tables(assignments, tile_table, tile_box_table)
save_tile_table(tile_table, config.paths.metrics_dir / "colab_tile_table.csv")
save_tile_box_table(tile_box_table, config.paths.metrics_dir / "colab_tile_box_table.csv")
tile_summary = summarize_tiles(tile_table, tile_box_table)
save_tile_summary(tile_summary, config.paths.reports_dir / "colab_tile_summary.json")

print(json.dumps({
    "trusted_images": len(trusted_records),
    "tiles": len(tile_table),
    "tile_boxes": len(tile_box_table),
    "fold_summary": fold_summary["folds"][str(FOLD_INDEX)],
    "seagull_risk": eda_summary["seagull_imbalance"],
}, indent=2))


In [ ]:
train_dataset, val_dataset = build_fold_tile_datasets(
    trusted_records,
    tile_table,
    tile_box_table,
    fold_index=FOLD_INDEX,
    tile_size=config.tiles.tile_size,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.train.batch_size,
    shuffle=True,
    num_workers=config.train.num_workers,
    pin_memory=True,
    collate_fn=detection_collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.train.batch_size,
    shuffle=False,
    num_workers=config.train.num_workers,
    pin_memory=True,
    collate_fn=detection_collate_fn,
)

anchor_generator = AnchorGenerator(
    sizes=config.anchors.sizes,
    aspect_ratios=config.anchors.aspect_ratios,
    strides=config.anchors.strides,
)
model = SmallObjectDetector(
    num_classes=len(config.data.classes),
    anchor_generator=anchor_generator,
    pretrained_backbone=config.model.pretrained,
    neck_channels=config.model.neck_channels,
    head_channels=config.model.head_channels,
).to(device)

if FREEZE_BACKBONE:
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False

optimizer = AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=config.train.learning_rate,
    weight_decay=config.train.weight_decay,
)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=config.train.scheduler_factor,
    patience=config.train.scheduler_patience,
)

print(f"Train tiles: {len(train_dataset)} | Val tiles: {len(val_dataset)}")


In [ ]:
training_summary = fit_detector(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_classes=len(config.data.classes),
    class_names=config.data.classes,
    epochs=config.train.epochs,
    checkpoint_dir=config.paths.checkpoints_dir,
    run_name=RUN_NAME,
    grad_clip_norm=config.train.grad_clip_norm,
    score_threshold=0.01,
    nms_threshold=0.5,
    topk_candidates=300,
    max_detections=50,
)
training_summary["fold_index"] = FOLD_INDEX
training_summary["device"] = str(device)
training_summary["frozen_backbone"] = FREEZE_BACKBONE
training_summary["train_tiles"] = len(train_dataset)
training_summary["val_tiles"] = len(val_dataset)
save_history(training_summary, config.paths.reports_dir / f"{RUN_NAME}_history.json")
print(json.dumps(training_summary, indent=2))


In [ ]:
best_epoch = training_summary["history"][-1]
print("Best checkpoint:", training_summary["best_checkpoint_path"])
print(json.dumps(best_epoch["val"], indent=2))

if USE_DRIVE_OUTPUTS:
    print(f"Artifacts saved under: {config.paths.outputs_dir}")
